# 02 · Matrix product operators and two-site DMRG

We will calculate the ground state of a six-spin Ising chain by changing two MPS tensors at a time. The Hamiltonian enters each local eigenvalue problem through contracted left and right environments. Increasing the retained bond dimension improves the variational state.

This notebook accompanies **Matrix Product Operators**, **Canonical forms**, and **Density matrix renormalization group** in the [lecture notes](https://arxiv.org/abs/2512.02706). It is self-contained and uses Python 3.10 or later with NumPy 1.26 or later. Run the cells in order. No tensor-network package is used.

We use dense matrices for the small *local* eigenvalue problems so that every index can be inspected. A separately constructed dense many-body Hamiltonian provides an exact-diagonalization comparison. It never enters a DMRG update.


In [1]:
import sys
import time
import numpy as np

assert sys.version_info >= (3, 10)
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
np.set_printoptions(precision=6, suppress=True)
started = time.perf_counter()
print("NumPy", np.__version__)


NumPy 2.5.3


## The physical system and the array convention

The open-chain Hamiltonian is
$$
H=-J\sum_{i=0}^{N-2}Z_i Z_{i+1}
  -h_x\sum_{i=0}^{N-1}X_i-h_z\sum_{i=0}^{N-1}Z_i.
$$
We use Pauli matrices with eigenvalues $\pm1$, rather than spin operators $S^\alpha=\sigma^\alpha/2$. Site zero is the **leftmost** factor in a Kronecker product. Flattening a state tensor with NumPy's default C order therefore gives the basis $|s_0s_1\cdots s_{N-1}\rangle$ with the last site varying fastest.

The arrays throughout the notebook have these shapes:

| Array | Axis meaning | Shape |
|---|---|---|
| `A[i][a,s,b]` | left MPS bond, physical index, right MPS bond | $(D_i,d,D_{i+1})$ |
| `W[i][mu,nu,s,t]` | left MPO bond, right MPO bond, bra/output, ket/input | $(w_i,w_{i+1},d,d)$ |
| `L[a,b,mu]` | bra MPS bond, ket MPS bond, MPO bond | $(D_i,D_i,w_i)$ |
| `R[c,d,rho]` | bra MPS bond, ket MPS bond, MPO bond | $(D_j,D_j,w_j)$ |
| `theta[a,s,t,c]` | left bond, first physical site, second physical site, right bond | $(D_i,d,d,D_{i+2})$ |

In an `einsum` string, repeated letters are summed unless they appear after `->`. Complex conjugation is written explicitly with `.conj()`.


In [2]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
N, J, hx, hz = 6, 1.0, 1.05, 0.08

# This dense construction sums the Pauli strings directly.
def operator_on_sites(N, insertions):
    result = np.ones((1, 1), dtype=complex)
    for site in range(N):
        result = np.kron(result, insertions.get(site, I))
    return result


def dense_ising(N, J, hx, hz):
    H = np.zeros((2**N, 2**N), dtype=complex)
    for i in range(N):
        H -= hx * operator_on_sites(N, {i: X})
        H -= hz * operator_on_sites(N, {i: Z})
    for i in range(N - 1):
        H -= J * operator_on_sites(N, {i: Z, i + 1: Z})
    return H

H_dense = dense_ising(N, J, hx, hz)
exact_energies, exact_vectors = np.linalg.eigh(H_dense)
E_exact, psi_exact = exact_energies[0], exact_vectors[:, 0]
print("Full Hilbert-space dimension:", H_dense.shape[0])
print(f"Exact ground-state energy: {E_exact:.12f}")


Full Hilbert-space dimension: 64
Exact ground-state energy: -7.682748637381


## Build the MPO from allowed operator strings

For a bulk site, take the operator-valued matrix
$$
W=\begin{pmatrix}
I&0&0\\ Z&0&0\\ -h_xX-h_zZ&-JZ&I
\end{pmatrix}.
$$
The first tensor keeps row 2 and the last keeps column 0, using zero-based array indices. Contracting auxiliary indices selects complete operator strings. A field term follows the transition $2\to0$ at one site. An interaction follows $2\to1\to0$ on adjacent sites and contributes $(-JZ)\otimes Z$. The identity entries propagate the unfinished or completed part of a string. There is no $1\to1$ entry, so the two $Z$ operators cannot separate.

The physical legs stay open when we contract the MPO's horizontal bonds. The upper legs below are ket inputs and the lower legs are bra outputs. Their order in memory remains `(mu, nu, s, t)`.




In [3]:
def ising_mpo(N, J, hx, hz):
    assert N >= 2
    W = np.zeros((3, 3, 2, 2), dtype=complex)
    W[0, 0] = I
    W[1, 0] = Z
    W[2, 0] = -hx * X - hz * Z
    W[2, 1] = -J * Z
    W[2, 2] = I
    tensors = [W.copy() for _ in range(N)]
    tensors[0] = W[2:3, :, :, :].copy()
    tensors[-1] = W[:, 0:1, :, :].copy()
    return tensors

mpo = ising_mpo(N, J, hx, hz)
print("MPO shapes:", [W.shape for W in mpo])


MPO shapes: [(1, 3, 2, 2), (3, 3, 2, 2), (3, 3, 2, 2), (3, 3, 2, 2), (3, 3, 2, 2), (3, 1, 2, 2)]


To check the MPO, let `partial[mu]` be the dense operator already assembled on the sites to the left. Appending a site gives
$$
\text{next}[\nu]=\sum_\mu \text{partial}[\mu]\otimes W_{\mu\nu}.
$$
This is only a small-system verification routine. DMRG will contract the MPO with MPS environments without assembling this dense operator.


In [4]:
def mpo_to_dense(mpo):
    partial = [np.ones((1, 1), dtype=complex)]
    for W in mpo:
        old_size = partial[0].shape[0]
        updated = []
        for nu in range(W.shape[1]):
            block = np.zeros((2 * old_size, 2 * old_size), dtype=complex)
            for mu in range(W.shape[0]):
                block += np.kron(partial[mu], W[mu, nu])
            updated.append(block)
        partial = updated
    assert len(partial) == 1
    return partial[0]

mpo_error = np.linalg.norm(mpo_to_dense(mpo) - H_dense)
assert mpo_error < 1e-12
print(f"||H_MPO - H_dense|| = {mpo_error:.3e}")


||H_MPO - H_dense|| = 6.181e-16


## Start with a canonical MPS

A normalized product state has tensors of shape $(1,2,1)$. Each tensor is both left- and right-canonical because $\sum_s|A^s|^2=1$. We deliberately use complex local vectors to test the conjugations in the contractions below.

For a tensor of larger bond dimension, left-canonical means
$$
\sum_{a,s}\overline{A_{asb}}A_{asb'}=\delta_{bb'},
$$
whereas right-canonical means
$$
\sum_{s,b}A_{asb}\overline{A_{a'sb}}=\delta_{aa'}.
$$
Around an active pair, all tensors to its left must satisfy the first condition and all tensors to its right the second. The pair itself contains the norm and the variational parameters.


In [5]:
def product_mps(N, seed=17):
    rng = np.random.default_rng(seed)
    tensors = []
    for _ in range(N):
        vector = rng.normal(size=2) + 1j * rng.normal(size=2)
        tensors.append((vector / np.linalg.norm(vector)).reshape(1, 2, 1))
    return tensors


def mps_to_dense(mps):
    state = np.ones((1, 1), dtype=complex)
    for A in mps:
        state = np.einsum("xa,asb->xsb", state, A).reshape(-1, A.shape[2])
    assert state.shape[1] == 1
    return state[:, 0]


def canonical_error(mps, bond):
    errors = [0.0]
    for A in mps[:bond]:
        M = A.reshape(-1, A.shape[2])
        errors.append(np.linalg.norm(M.conj().T @ M - np.eye(M.shape[1])))
    for A in mps[bond + 2:]:
        M = A.reshape(A.shape[0], -1)
        errors.append(np.linalg.norm(M @ M.conj().T - np.eye(M.shape[0])))
    return max(errors)

initial_mps = product_mps(N)
assert abs(np.linalg.norm(mps_to_dense(initial_mps)) - 1) < 1e-12
print("Initial MPS shapes:", [A.shape for A in initial_mps])
print("Canonical error around bond 0:", canonical_error(initial_mps, 0))


Initial MPS shapes: [(1, 2, 1), (1, 2, 1), (1, 2, 1), (1, 2, 1), (1, 2, 1), (1, 2, 1)]
Canonical error around bond 0: 2.220446049250313e-16


## Contract one site into an environment

The book orders the MPS indices of each environment as **bra, then ket**. With the code's letters, absorbing one site into the left block gives
$$
L^{\rm next}_{uv\nu}
 =\sum_{a,b,\mu,s,t}\overline{A_{asu}}\,
 L_{ab\mu}\,W_{\mu\nu st}\,A_{btv}.
$$
The ket tensor is above the operator in the diagram. Its conjugate is below. The three legs on the right are precisely the three axes of `Lnext`.



Absorbing into the right block sums the same tensors in the other direction:
$$
R^{\rm next}_{ab\mu}
 =\sum_{u,v,\nu,s,t}\overline{A_{asu}}\,
 W_{\mu\nu st}\,A_{btv}\,R_{uv\nu}.
$$
The environment for an empty block is the scalar one, stored as a `(1,1,1)` array. These formulas work for arbitrary complex MPS tensors, whether or not they are canonical.


In [6]:
def absorb_left(L, A, W):
    return np.einsum("asu,abm,mnst,btv->uvn", A.conj(), L, W, A,
                     optimize=True)


def absorb_right(R, A, W):
    return np.einsum("asu,mnst,btv,uvn->abm", A.conj(), W, A, R,
                     optimize=True)


def left_environments(mps, mpo):
    L = [np.ones((1, 1, 1), dtype=complex)]
    for A, W in zip(mps, mpo):
        L.append(absorb_left(L[-1], A, W))
    return L


def right_environments(mps, mpo):
    R = [None] * (len(mps) + 1)
    R[-1] = np.ones((1, 1, 1), dtype=complex)
    for i in range(len(mps) - 1, -1, -1):
        R[i] = absorb_right(R[i + 1], mps[i], mpo[i])
    return R


def expectation_mpo(mps, mpo):
    return left_environments(mps, mpo)[-1].item()

L0 = left_environments(initial_mps, mpo)
R0 = right_environments(initial_mps, mpo)
psi0 = mps_to_dense(initial_mps)
E0_dense = np.vdot(psi0, H_dense @ psi0)
assert np.allclose(L0[-1].item(), E0_dense, atol=1e-12)
assert np.allclose(R0[0].item(), E0_dense, atol=1e-12)
print("Energy from left contraction :", L0[-1].item())
print("Energy from right contraction:", R0[0].item())


Energy from left contraction : (-1.2951864718926773+0j)
Energy from right contraction: (-1.2951864718926775+0j)


## The two-site effective Hamiltonian

Keep the surrounding MPS tensors fixed and vary
$$
\Theta_{astc}=\sum_b A^{[i]}_{asb}A^{[i+1]}_{btc}.
$$
The outside canonical tensors define orthonormal block states. Therefore $\|\psi(\Theta)\|^2=\sum_{astc}|\Theta_{astc}|^2$, and the local energy is an ordinary Rayleigh quotient.

The effective matrix has elements
$$
(H_{\rm eff})_{asuc,\,btvd}
 =\sum_{\mu,\nu,\rho}
 L_{ab\mu}W^{[i]}_{\mu\nu st}
 W^{[i+1]}_{\nu\rho uv}R_{cd\rho}.
$$
The first four axes form the matrix row and the last four form its column. Applying the matrix contracts the column with $\Theta_{btvd}$ and leaves the four output legs shown here.



For this six-site demonstration the matrix has at most ${64}$ rows. Storing it costs $O(m^2)$ and dense diagonalization costs $O(m^3)$ for $m=d^2D_iD_{i+2}$. Larger calculations use an iterative eigensolver and contract the action directly. The next cell implements both forms and checks that they agree.


In [7]:
def effective_matrix(L, W1, W2, R):
    tensor = np.einsum("abm,mnst,nruv,cdr->asucbtvd", L, W1, W2, R,
                       optimize=True)
    shape = (L.shape[0], W1.shape[2], W2.shape[2], R.shape[0])
    size = int(np.prod(shape))
    matrix = tensor.reshape(size, size)
    assert np.linalg.norm(matrix - matrix.conj().T) < 1e-10
    return matrix, shape


def effective_action(theta, L, W1, W2, R):
    return np.einsum("abm,mnst,nruv,cdr,btvd->asuc", L, W1, W2, R,
                     theta, optimize=True)

H_eff, pair_shape = effective_matrix(L0[0], mpo[0], mpo[1], R0[2])
rng = np.random.default_rng(21)
trial = rng.normal(size=pair_shape) + 1j * rng.normal(size=pair_shape)
action = effective_action(trial, L0[0], mpo[0], mpo[1], R0[2])
assert np.allclose(action.ravel(), H_eff @ trial.ravel(), atol=1e-12)
print("Active tensor shape:", pair_shape)
print("Effective matrix shape:", H_eff.shape)


Active tensor shape: (1, 2, 2, 1)
Effective matrix shape: (4, 4)


We can also verify the meaning of this matrix independently of the MPO contraction. Let $Q$ embed the local tensor coordinates into the full Hilbert space by attaching the fixed left and right MPS blocks. Then
$$
Q^\dagger Q=I,\qquad H_{\rm eff}=Q^\dagger H Q.
$$
The first equality checks the canonical metric. The second checks all bra, ket, physical, and auxiliary index placements in the effective Hamiltonian. `Q` is only used for this verification.


In [8]:
def pair_embedding(mps, bond):
    left = np.ones((1, 1), dtype=complex)
    for A in mps[:bond]:
        left = np.einsum("xa,asb->xsb", left, A).reshape(-1, A.shape[2])
    right = np.ones((1, 1), dtype=complex)
    for A in reversed(mps[bond + 2:]):
        right = np.einsum("asb,bx->asx", A, right).reshape(A.shape[0], -1)
    tensor = np.einsum("xa,su,tv,cy->xstyauvc", left, I, I, right)
    return tensor.reshape(2**len(mps), -1)

Q = pair_embedding(initial_mps, 0)
assert np.allclose(Q.conj().T @ Q, np.eye(Q.shape[1]), atol=1e-12)
projection_error = np.linalg.norm(H_eff - Q.conj().T @ H_dense @ Q)
assert projection_error < 1e-11
print(f"||H_eff - Q† H_dense Q|| = {projection_error:.3e}")


||H_eff - Q† H_dense Q|| = 1.722e-15


## Optimize the pair, then choose the retained Schmidt space

For the normalized old pair vector $\theta$, the current local energy is $\theta^\dagger H_{\rm eff}\theta$. The smallest eigenvector minimizes this quotient because the block metric is the identity. Before truncation, its eigenvalue cannot exceed the old energy.

After finding the new pair, reshape it as $\Theta_{(a,s),(t,c)}$ and perform an SVD. Retaining the largest $\chi$ singular values gives discarded weight
$$
\epsilon_{\rm disc}=\sum_{r\geq\chi}\sigma_r^2.
$$
The optimized pair has unit norm, so the retained state's squared norm is $1-\epsilon_{\rm disc}$. We divide its retained singular values by $\sqrt{1-\epsilon_{\rm disc}}$ to normalize it.



The diagram uses `S` for the diagonal singular-value matrix $\Sigma$ and `Vh` for $V^\dagger$, as in the code's SVD output.

On a **right sweep**, store $A=U$ and $B=\Sigma V^\dagger$. Site $i$ becomes left-canonical and the center moves to $i+1$. On a **left sweep**, store $A=U\Sigma$ and $B=V^\dagger$. Site $i+1$ becomes right-canonical and the center moves to $i$. No inverse Schmidt values are needed.


In [9]:
def lowest_pair(matrix, shape):
    energies, vectors = np.linalg.eigh(matrix)
    energy = energies[0]
    theta = vectors[:, 0].reshape(shape)
    residual = np.linalg.norm(matrix @ theta.ravel() - energy * theta.ravel())
    return theta, energy, residual


def split_pair(theta, chi, direction):
    Dl, d1, d2, Dr = theta.shape
    U, singular, Vh = np.linalg.svd(theta.reshape(Dl * d1, d2 * Dr),
                                   full_matrices=False)
    keep = min(chi, len(singular))
    discarded = float(np.sum(singular[keep:]**2))
    U, singular, Vh = U[:, :keep], singular[:keep], Vh[:keep]
    singular = singular / np.linalg.norm(singular)
    if direction == "right":
        A = U.reshape(Dl, d1, keep)
        B = (singular[:, None] * Vh).reshape(keep, d2, Dr)
    elif direction == "left":
        A = (U * singular[None, :]).reshape(Dl, d1, keep)
        B = Vh.reshape(keep, d2, Dr)
    else:
        raise ValueError("direction must be 'right' or 'left'")
    return A, B, discarded

optimized, local_energy, local_residual = lowest_pair(H_eff, pair_shape)
A, B, discarded = split_pair(optimized, chi=2, direction="right")
reconstructed = np.einsum("asb,btc->astc", A, B)
assert np.allclose(reconstructed, optimized, atol=1e-12)
assert np.allclose(A.reshape(-1, A.shape[2]).conj().T @
                   A.reshape(-1, A.shape[2]), np.eye(A.shape[2]), atol=1e-12)
print(f"First local minimum: {local_energy:.12f}")
print(f"Local eigenvector residual: {local_residual:.3e}")
print("New tensor shapes:", A.shape, B.shape)


First local minimum: -2.511538391294
Local eigenvector residual: 4.606e-16
New tensor shapes: (1, 2, 2) (2, 2, 1)


The update routine below joins these operations. It records the energy **before the solve**, **after the solve**, and **after truncation**. The solve lowers the Rayleigh quotient. The SVD chooses the best approximation in state norm, so truncation can raise the energy again. These are distinct steps.

The routine receives only the two environments, the two MPO tensors, and the current MPS. It has no access to `H_dense` or the exact ground state.


In [10]:
def update_pair(mps, mpo, bond, L, R, chi, direction):
    assert canonical_error(mps, bond) < 1e-10
    matrix, shape = effective_matrix(L, mpo[bond], mpo[bond + 1], R)
    old = np.einsum("asb,btc->astc", mps[bond], mps[bond + 1]).ravel()
    before = float(np.vdot(old, matrix @ old).real / np.vdot(old, old).real)
    theta, minimum, residual = lowest_pair(matrix, shape)
    assert minimum <= before + 1e-10
    A, B, discarded = split_pair(theta, chi, direction)
    mps[bond], mps[bond + 1] = A, B
    kept = np.einsum("asb,btc->astc", A, B).ravel()
    after = float(np.vdot(kept, matrix @ kept).real)
    assert after >= minimum - 1e-10
    return {"bond": bond, "direction": direction, "before": before,
            "minimum": minimum, "after": after, "discarded": discarded,
            "residual": residual, "local_size": len(old)}


## Move across the chain and back

At the start of a right sweep, cache all right environments. After optimizing bond $(i,i+1)$, absorb the new left-canonical tensor at $i$ into the left environment. The cached right block begins at $i+2$ and has not been changed yet. Its environment remains valid.

On the return sweep, cache the left environments once and build the right environment as the center moves left. Every local solve therefore uses the current optimized block behind the center and the unchanged block ahead of it. The two functions show this motion directly.


In [11]:
def sweep_right(mps, mpo, chi):
    right = right_environments(mps, mpo)
    left = np.ones((1, 1, 1), dtype=complex)
    records = []
    for bond in range(len(mps) - 1):
        records.append(update_pair(mps, mpo, bond, left, right[bond + 2],
                                   chi, "right"))
        left = absorb_left(left, mps[bond], mpo[bond])
    return records


def sweep_left(mps, mpo, chi):
    left = left_environments(mps, mpo)
    right = np.ones((1, 1, 1), dtype=complex)
    records = []
    for bond in range(len(mps) - 2, -1, -1):
        records.append(update_pair(mps, mpo, bond, left[bond], right,
                                   chi, "left"))
        right = absorb_right(right, mps[bond + 1], mpo[bond + 1])
    return records

# Inspect one round trip before using the repeated-sweep driver.
example_mps = product_mps(N)
example_records = sweep_right(example_mps, mpo, chi=2)
example_records += sweep_left(example_mps, mpo, chi=2)
for entry in example_records:
    print(f"{entry['direction']:5s} bond {entry['bond']}  "
          f"E_before={entry['before']: .8f}  E_min={entry['minimum']: .8f}  "
          f"E_kept={entry['after']: .8f}  discarded={entry['discarded']:.2e}")
assert abs(np.linalg.norm(mps_to_dense(example_mps)) - 1) < 1e-11


right bond 0  E_before=-1.29518647  E_min=-2.51153839  E_kept=-2.51153839  discarded=0.00e+00
right bond 1  E_before=-2.51153839  E_min=-4.09879567  E_kept=-4.09879567  discarded=0.00e+00
right bond 2  E_before=-4.09879567  E_min=-7.00706968  E_kept=-7.00706968  discarded=0.00e+00
right bond 3  E_before=-7.00706968  E_min=-7.43421082  E_kept=-7.43421082  discarded=0.00e+00
right bond 4  E_before=-7.43421082  E_min=-7.67862708  E_kept=-7.67862708  discarded=0.00e+00
left  bond 4  E_before=-7.67862708  E_min=-7.67862708  E_kept=-7.67862708  discarded=0.00e+00
left  bond 3  E_before=-7.67862708  E_min=-7.67887005  E_kept=-7.67869933  discarded=2.88e-05
left  bond 2  E_before=-7.67869933  E_min=-7.68182954  E_kept=-7.68151704  discarded=5.58e-05
left  bond 1  E_before=-7.68151704  E_min=-7.68214675  E_kept=-7.68192613  discarded=3.79e-05
left  bond 0  E_before=-7.68192613  E_min=-7.68193203  E_kept=-7.68193203  discarded=0.00e+00


A round trip ends with the center on site zero, ready for the next right sweep. The driver below records one energy per round trip. It does not assume that truncation makes each individual update monotonic.

We run $\chi=1$, ${2}$, and ${8}$. A six-spin state has Schmidt rank at most $2^3=8$, so the last run can represent the exact ground state. The $\chi=1$ run retains product states. Its two-site solve still explores an entangled pair before truncating back to rank one, so it should not be interpreted as an exact solver for the best product-state energy.


In [12]:
def dmrg(mpo, chi, sweeps=8, seed=17):
    assert chi >= 1 and sweeps >= 1
    mps = product_mps(len(mpo), seed=seed)
    history, records = [], []
    for sweep in range(sweeps):
        steps = sweep_right(mps, mpo, chi)
        steps += sweep_left(mps, mpo, chi)
        records.extend(steps)
        energy = float(expectation_mpo(mps, mpo).real)
        history.append(energy)
        assert max(A.shape[2] for A in mps) <= chi
    return {"mps": mps, "energy": history[-1], "history": np.array(history),
            "records": records}

runs = {chi: dmrg(mpo, chi) for chi in (1, 2, 8)}
print("Round trip      chi=1               chi=2               chi=8")
for sweep in range(8):
    energies = " ".join(f"{runs[chi]['history'][sweep]: .12f}" for chi in runs)
    print(f"{sweep + 1:5d}       {energies}")


Round trip      chi=1               chi=2               chi=8
    1       -7.441552306975 -7.681932030593 -7.682748577492
    2       -7.443344371228 -7.681928252786 -7.682748637381
    3       -7.443359026870 -7.681928202230 -7.682748637381
    4       -7.443359165834 -7.681928202068 -7.682748637381
    5       -7.443359167186 -7.681928202068 -7.682748637381
    6       -7.443359167199 -7.681928202068 -7.682748637381
    7       -7.443359167199 -7.681928202068 -7.682748637381
    8       -7.443359167199 -7.681928202068 -7.682748637381


## Compare energy, state overlap, and a local observable

The energy is variational, but energy alone does not specify the state error. We also compute the fidelity $|\langle\psi_{\rm exact}|\psi_\chi\rangle|^2$ and the central spin expectation $\langle X_{N/2}\rangle$.

A local observable can use the same environment contraction: put the observable at its site and an identity at every other site. The resulting MPO has auxiliary dimension one. We compare its contraction with an independent dense expectation value.


In [13]:
def local_observable(mps, site, operator):
    local_mpo = [I.reshape(1, 1, 2, 2).copy() for _ in mps]
    local_mpo[site] = operator.reshape(1, 1, 2, 2)
    return expectation_mpo(mps, local_mpo)

center = N // 2
X_dense = operator_on_sites(N, {center: X})
X_exact = float(np.vdot(psi_exact, X_dense @ psi_exact).real)
print("chi     E - E_exact       fidelity          <X_center>     max local residual")
for chi, result in runs.items():
    state = mps_to_dense(result["mps"])
    fidelity = abs(np.vdot(psi_exact, state))**2
    spin = local_observable(result["mps"], center, X)
    dense_energy = np.vdot(state, H_dense @ state).real
    residual = max(r["residual"] for r in result["records"])
    assert abs(np.linalg.norm(state) - 1) < 1e-11
    assert abs(dense_energy - result["energy"]) < 1e-10
    assert np.allclose(spin, np.vdot(state, X_dense @ state), atol=1e-11)
    assert result["energy"] >= E_exact - 1e-10
    result.update(fidelity=float(fidelity), spin=float(spin.real))
    print(f"{chi:3d}   {result['energy'] - E_exact: .3e}    {fidelity:.10f}    "
          f"{spin.real:.10f}       {residual:.2e}")
assert abs(runs[8]["energy"] - E_exact) < 1e-10
assert runs[8]["fidelity"] > 1 - 1e-10
assert abs(runs[8]["spin"] - X_exact) < 1e-10
assert runs[2]["energy"] < runs[1]["energy"]
print(f"Exact central <X>: {X_exact:.10f}")


chi     E - E_exact       fidelity          <X_center>     max local residual
  1    2.394e-01    0.8573346041    0.5395303446       2.90e-15
  2    8.204e-04    0.9998126699    0.6570280659       4.95e-15
  8   -1.066e-14    1.0000000000    0.6572189594       1.17e-14
Exact central <X>: 0.6572189594


One further check uses an internal bond after the algorithm has built nontrivial Schmidt spaces. The completed round trip has its center on site zero. Moving right through the first two bonds puts the center beside the chosen internal pair. We can then test $Q^\dagger Q=I$ and $H_{\rm eff}=Q^\dagger H Q$ in the enlarged local basis.


In [14]:
check_mps = [A.copy() for A in runs[8]["mps"]]
right = right_environments(check_mps, mpo)
left = np.ones((1, 1, 1), dtype=complex)
for bond in range(2):
    update_pair(check_mps, mpo, bond, left, right[bond + 2], 8, "right")
    left = absorb_left(left, check_mps[bond], mpo[bond])
internal = 2
matrix, shape = effective_matrix(left, mpo[internal], mpo[internal + 1],
                                 right[internal + 2])
Q = pair_embedding(check_mps, internal)
assert np.allclose(Q.conj().T @ Q, np.eye(Q.shape[1]), atol=2e-11)
assert np.allclose(matrix, Q.conj().T @ H_dense @ Q, atol=2e-11)
assert np.allclose(effective_action(np.ones(shape, dtype=complex), left,
                                   mpo[internal], mpo[internal + 1],
                                   right[internal + 2]).ravel(),
                   matrix @ np.ones(int(np.prod(shape))), atol=2e-11)
print("Internal pair shape:", shape)
print("Internal projection error:", np.linalg.norm(matrix - Q.conj().T @ H_dense @ Q))
print(f"Notebook calculation time: {time.perf_counter() - started:.3f} s")


Internal pair shape: (4, 2, 2, 4)
Internal projection error: 6.413892539461683e-15
Notebook calculation time: 1.482 s


## Exercises

1. **Decoupled spins.** Set $J=0$ while keeping the fields above. Predict $E_0=-N\sqrt{h_x^2+h_z^2}$ and $\langle X\rangle=h_x/\sqrt{h_x^2+h_z^2}$. Run `dmrg(ising_mpo(N, 0, hx, hz), chi=1)` and compare both quantities. The ground state factorizes, so no bond dimension above one is needed. A completed check is included below.

2. **Energy and correlation accuracy.** Repeat the three bond-dimension runs at $h_x=0.5$ and $h_x=1.5$, keeping $J=1$ and $h_z=0.08$. Rebuild the independent dense Hamiltonian each time. Compare energy error, fidelity, and the connected central correlation $\langle Z_iZ_{i+1}\rangle-\langle Z_i\rangle\langle Z_{i+1}\rangle$. Use a dimension-one observable MPO with $Z$ at both sites to obtain the first term. Explain why every product state has zero connected correlation, even if its energy error is modest.

3. **Inspect a discarded Schmidt sector.** In the first $\chi=2$ round trip, identify the update with the largest `discarded`. Rerun with $\chi=4$ and compare that weight, the final energy, and the final fidelity. Check each record's `minimum <= before` and compare `after - minimum` with the discarded weight. The energy increase depends on the Hamiltonian as well as the lost norm, so these two quantities need not coincide.


In [15]:
# A completed exercise check. This cell is part of Run All.
free_spins = dmrg(ising_mpo(N, 0.0, hx, hz), chi=1, sweeps=2)
field_size = np.sqrt(hx**2 + hz**2)
assert abs(free_spins["energy"] + N * field_size) < 1e-11
assert abs(local_observable(free_spins["mps"], center, X).real - hx / field_size) < 1e-11
print(f"J=0, chi=1: energy {free_spins['energy']:.12f}")
print("All MPO, environment, canonical-form, and DMRG comparison checks passed.")


J=0, chi=1: energy -6.318259253940
All MPO, environment, canonical-form, and DMRG comparison checks passed.
